# [스프린트 미션 5] Baseline — CNN 오토인코더로 문서 복원하기 (+심화: ECG 이상 탐지)

- **Part 1 (기본 미션)**: 손상된 문서 이미지를 복원하는 CNN 오토인코더
- **Part 2 (심화, 채점 제외)**: 정상 심전도만 학습해 이상 심박을 찾아내는 LSTM 오토인코더


In [ ]:
import os
import random
# dlkfjdlkfjsldkfjs

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from PIL import Image
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("PyTorch:", torch.__version__, "| device:", device)

PyTorch: 2.11.0+cu128 | device: cuda


---
# Part 1. 손상된 문서 복원 — CNN 오토인코더

train(손상 이미지)과 train_cleaned(원본 이미지)를 한 쌍으로 학습합니다.
손상 이미지를 입력하면 원본에 가까운 이미지를 출력하도록 인코더-디코더를 학습시키는 구조예요.

## 1-1. 데이터 준비
Kaggle에서 데이터를 내려받습니다. 두 가지 준비가 필요해요.
1. [대회 페이지](https://www.kaggle.com/competitions/denoising-dirty-documents)에서 **Join Competition**을 눌러 규칙에 동의합니다. (동의하지 않으면 다운로드가 거부됩니다)
2. Kaggle > Settings > API에서 **Create New Token**으로 `kaggle.json`을 발급받아 두세요.

In [2]:
# kaggle.json을 업로드하세요.
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c denoising-dirty-documents
!unzip -q denoising-dirty-documents.zip -d data
!cd data && unzip -q train.zip && unzip -q train_cleaned.zip && unzip -q test.zip
!ls data

KeyboardInterrupt: 

## 1-2. Dataset 구성
이미지 크기가 두 종류(420×540, 258×540)라서, 학습할 때는 **같은 위치에서 128×128 패치를 잘라** 손상/원본 쌍으로 사용합니다.
패치 단위로 학습하면 이미지 크기를 신경 쓰지 않아도 되고, 한 이미지에서 매번 다른 부분을 보게 되어 데이터가 늘어나는 효과도 있어요.

In [3]:
class DirtyDocsDataset(Dataset):
    """손상 이미지와 원본 이미지에서 같은 위치의 패치를 한 쌍으로 돌려줍니다."""

    def __init__(self, dirty_dir, clean_dir, crop_size=128):
        self.dirty_dir = dirty_dir
        self.clean_dir = clean_dir
        self.files = sorted(os.listdir(dirty_dir))
        self.crop_size = crop_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        dirty = Image.open(os.path.join(self.dirty_dir, name)).convert("L")
        clean = Image.open(os.path.join(self.clean_dir, name)).convert("L")

        # 두 이미지에서 '같은 위치'를 잘라야 하므로 crop 좌표를 직접 뽑습니다.
        cs = self.crop_size
        top = random.randint(0, dirty.height - cs)
        left = random.randint(0, dirty.width - cs)
        dirty = TF.to_tensor(TF.crop(dirty, top, left, cs, cs))  # (1, 128, 128), 0~1
        clean = TF.to_tensor(TF.crop(clean, top, left, cs, cs))
        return dirty, clean


train_dataset = DirtyDocsDataset("data/train", "data/train_cleaned")
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print("훈련 이미지 수:", len(train_dataset))

FileNotFoundError: [Errno 2] No such file or directory: 'data/train'

In [ ]:
# 손상/원본 쌍이 잘 만들어졌는지 확인해 봅시다.
dirty, clean = train_dataset[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(dirty.squeeze(), cmap="gray"); axes[0].set_title("dirty")
axes[1].imshow(clean.squeeze(), cmap="gray"); axes[1].set_title("clean")
for ax in axes: ax.axis("off")
plt.show()

## 1-3. CNN 오토인코더 설계
`Conv2d`(stride=2)로 크기를 절반씩 줄이며 특징을 압축하고, `ConvTranspose2d`로 다시 원래 크기까지 복원합니다.
층마다 텐서 크기가 어떻게 변하는지 주석으로 적어 두세요. 인코더와 디코더의 크기가 어긋나는 문제가 이 미션에서 가장 자주 만나는 에러입니다.

In [ ]:
class ConvDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),   # (1, 128, 128) -> (32, 64, 64)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # (32, 64, 64) -> (64, 32, 32)
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # (64, 32, 32) -> (32, 64, 64)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),   # (32, 64, 64) -> (1, 128, 128)
            nn.Sigmoid(),  # 픽셀 값을 0~1 범위로
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = ConvDenoiser().to(device)
print(model)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for dirty, clean in train_loader:
        dirty, clean = dirty.to(device), clean.to(device)
        optimizer.zero_grad()
        restored = model(dirty)
        loss = criterion(restored, clean)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(dirty)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1:3d} | loss {epoch_loss / len(train_dataset):.5f}")

## 1-4. 전체 이미지 복원과 평가
학습은 128×128 패치로 했지만, 합성곱 모델은 입력 크기가 달라도 동작합니다.
다만 stride=2 층을 두 번 지나므로 가로/세로가 4의 배수여야 해요. 4의 배수가 아닌 이미지(258×540)는 가장자리를 패딩했다가 복원 후 잘라냅니다.

In [ ]:
def denoise_full_image(model, img_tensor, multiple=4):
    """크기가 4의 배수가 아니면 오른쪽/아래를 패딩한 뒤 복원하고, 원래 크기로 되돌립니다."""
    _, h, w = img_tensor.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    x = F.pad(img_tensor.unsqueeze(0), (0, pad_w, 0, pad_h), mode="reflect")
    model.eval()
    with torch.no_grad():
        out = model(x.to(device)).cpu()
    return out[0, :, :h, :w]


# 훈련 이미지 하나를 통째로 복원해 봅니다.
name = train_dataset.files[0]
dirty_full = TF.to_tensor(Image.open(f"data/train/{name}").convert("L"))
clean_full = TF.to_tensor(Image.open(f"data/train_cleaned/{name}").convert("L"))
restored_full = denoise_full_image(model, dirty_full)

rmse = torch.sqrt(((restored_full - clean_full) ** 2).mean())
print(f"RMSE: {rmse:.4f}")

fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for ax, img, title in zip(axes, [dirty_full, restored_full, clean_full], ["dirty", "restored", "clean"]):
    ax.imshow(img.squeeze(), cmap="gray"); ax.set_title(title); ax.axis("off")
plt.show()

## 1-5. 여기서부터는 여러분 차례입니다
Baseline은 얕은 오토인코더라 흐릿한 얼룩 정도만 지웁니다. 아래 방향들을 실험하며 복원 품질을 높여 보세요.
- 인코더/디코더를 더 깊게 쌓거나 채널 수를 늘려 보기 (BatchNorm 추가 포함)
- 인코더의 중간 출력을 디코더에 이어 주는 스킵 커넥션 추가해 보기
- 좌우 반전, 밝기 조절 같은 데이터 증강 적용해 보기
- 검증 세트를 나눠 과적합 여부를 확인하고, test 이미지 복원 결과 비교하기
- 평가 지표에 PSNR을 추가하고, 복원이 잘 안 되는 손상 유형 분석하기

---
# Part 2. (심화) 심전도 이상 탐지 — LSTM 오토인코더

심화 문제는 채점에 포함되지 않아요. 여유가 될 때 도전해 봅시다.

이번에는 순서가 있는 데이터입니다. **정상 심전도(ECG) 신호만으로** LSTM 오토인코더를 학습시킵니다.
정상 패턴만 배운 모델은 이상 신호를 제대로 복원하지 못하므로, **복원 오차가 큰 신호를 이상 심박으로 판정**할 수 있어요.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv", header=None)
data = df.iloc[:, :-1].values.astype("float32")  # (4998, 140) — 신호 하나당 140 타임스텝
labels = df.iloc[:, -1].values.astype(int)       # 1: 정상, 0: 이상

X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42, stratify=labels
)

# 훈련 데이터 기준으로 0~1 정규화
mn, mx = X_train.min(), X_train.max()
X_train = (X_train - mn) / (mx - mn)
X_test = (X_test - mn) / (mx - mn)

# 핵심: 정상 신호만 골라 학습 세트를 만듭니다.
X_train_normal = X_train[y_train == 1]
print("전체 훈련:", X_train.shape, "| 정상만:", X_train_normal.shape, "| 테스트:", X_test.shape)

In [ ]:
# 정상 신호와 이상 신호를 눈으로 비교해 봅시다.
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(X_train[y_train == 1][0]); axes[0].set_title("normal")
axes[1].plot(X_train[y_train == 0][0]); axes[1].set_title("anomaly")
plt.show()

## 2-1. LSTM 오토인코더 설계
인코더 LSTM의 **마지막 hidden state**가 신호 전체를 요약한 잠재 벡터가 됩니다.
디코더는 이 벡터를 타임스텝 수만큼 반복해 입력받아, 원래 신호를 되살려 냅니다.

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len=140, n_features=1, hidden_size=64):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        # x: (batch, 140, 1)
        _, (h, _) = self.encoder(x)                          # h: (1, batch, 64) — 신호 전체의 요약
        z = h[-1]                                            # (batch, 64)
        dec_in = z.unsqueeze(1).repeat(1, self.seq_len, 1)   # (batch, 140, 64)
        out, _ = self.decoder(dec_in)                        # (batch, 140, 64)
        return self.output_layer(out)                        # (batch, 140, 1)


ae = LSTMAutoencoder().to(device)
print(ae)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X_train_normal_t = torch.tensor(X_train_normal).unsqueeze(-1)  # (N, 140, 1)
normal_loader = DataLoader(TensorDataset(X_train_normal_t), batch_size=128, shuffle=True)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)

EPOCHS = 100
for epoch in range(EPOCHS):
    ae.train()
    epoch_loss = 0.0
    for (xb,) in normal_loader:
        xb = xb.to(device)
        optimizer.zero_grad()
        loss = criterion(ae(xb), xb)  # 입력을 그대로 복원하도록 학습
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch + 1:3d} | loss {epoch_loss / len(X_train_normal_t):.5f}")

## 2-2. 복원 오차 분포와 임계값
정상 훈련 신호의 복원 오차 분포를 기준으로 임계값을 정합니다.
Baseline은 `평균 + 2×표준편차`를 사용했지만, 백분위수(예: 95%) 등 다른 기준도 실험해 보세요.

In [ ]:
def recon_errors(model, X):
    """신호별 복원 오차(MSE)를 계산합니다."""
    model.eval()
    x = torch.tensor(X).unsqueeze(-1)
    with torch.no_grad():
        pred = model(x.to(device)).cpu()
    return ((pred - x) ** 2).mean(dim=(1, 2)).numpy()


train_errors = recon_errors(ae, X_train_normal)
threshold = train_errors.mean() + 2 * train_errors.std()
print(f"임계값: {threshold:.5f}")

# 테스트 세트에서 정상/이상의 오차 분포가 갈라지는지 확인해 봅시다.
test_errors = recon_errors(ae, X_test)
plt.figure(figsize=(8, 4))
plt.hist(test_errors[y_test == 1], bins=50, alpha=0.6, label="normal")
plt.hist(test_errors[y_test == 0], bins=50, alpha=0.6, label="anomaly")
plt.axvline(threshold, color="red", linestyle="--", label="threshold")
plt.xlabel("reconstruction error"); plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

# 오차가 임계값보다 크면 이상(0), 작으면 정상(1)으로 판정합니다.
y_pred = (test_errors <= threshold).astype(int)
print(classification_report(y_test, y_pred, target_names=["anomaly(0)", "normal(1)"], digits=3))

In [ ]:
# 모델이 이상 신호를 '못' 복원하는 모습을 직접 확인해 봅시다.
def plot_reconstruction(model, signal, title):
    x = torch.tensor(signal).reshape(1, -1, 1)
    with torch.no_grad():
        pred = model(x.to(device)).cpu().squeeze()
    plt.plot(signal, label="original")
    plt.plot(pred, label="reconstructed")
    plt.fill_between(range(len(signal)), signal, pred, alpha=0.3)
    plt.title(title); plt.legend()


plt.figure(figsize=(12, 3.5))
plt.subplot(1, 2, 1); plot_reconstruction(ae, X_test[y_test == 1][0], "normal")
plt.subplot(1, 2, 2); plot_reconstruction(ae, X_test[y_test == 0][0], "anomaly")
plt.show()

## 2-3. 더 실험해 볼 것들
- 임계값 기준을 바꿔 보며 Precision과 Recall이 어떻게 움직이는지 관찰하기
- hidden size, 층 수를 조절하거나 LSTM을 GRU로 바꿔 비교하기
- 인코더를 양방향(bidirectional)으로 바꾸면 복원 품질이 달라지는지 확인하기

기본 미션과 심화 문제 모두 하나의 노트북에 담아 `05_{팀명}_{성함}.ipynb`로 제출해 주세요.